# Цель проекта:
###  - Проанализировать транзакционные данные интернет‑магазина (датасет Online Retail) и построить ML‑модели для прогноза вероятности покупки и её суммы в течение месяца на уровне клиента.
### - Выявить ключевые факторы покупательского поведения (частота заказов, доля возвратов, средний чек и др.) и сформировать сегменты клиентов, чтобы связать аналитические выводы с бизнес‑рекомендациями.

# Задачи проекта:
### 1. Загрузка, первичный EDA сырых транзакций и подготовка типов данных
### 2. Очистка данных от выбросов
### 3. Агрегация данных на уровне клиента (Customer‑level features)
### 4. Поиск скрытых корреляций
### 5. Сегментация клиентов и формирование «портрета потребителя»
### 6. Построение ML‑модели для прогноза вероятности покупки и прогноза выручки
### 7. Интерпретация результатов и связь с бизнес‑выводами

### 1. Загрузка и первичный EDA сырых транзакций

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Настройка отображения таблиц
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_rows', 50)
pd.options.mode.chained_assignment = None

# Загрузка и переименование столбцов
df = pd.read_csv('Online_retail.csv', sep=';',compression='gzip')
original_df = df.copy()

df = df.rename(columns={
    'InvoiceNo': 'Номер_заказа',
    'StockCode': 'Код_Товара',
    'Description': 'Товар',
    'Quantity': 'Количество',
    'InvoiceDate': 'Дата_покупки',
    'UnitPrice': 'Цена_шт',
    'CustomerID': 'ID_покупателя',
    'Country': 'Страна'
})

df.info()

# Удаление пропусков
df.dropna(inplace=True)

# Удаление дубликатов
df.drop_duplicates(inplace=True)

df.info()



In [ ]:
# Анализ типов данных: доля числовых значений по столбцам
for col in df.columns:
    numeric_mask = pd.to_numeric(df[col], errors='coerce').notna()
    percent_numeric = numeric_mask.mean() * 100
    print(f"В колонке '{col}' {percent_numeric:.1f}% значений — числа (или выглядят как числа)")

# Проверка корректности дат в колонке «Дата_покупки»
date_col = 'Дата_покупки'
date_mask = pd.to_datetime(df[date_col], errors='coerce').notna()
percent_date = date_mask.mean() * 100
print(f"В колонке '{date_col}' {percent_date:.1f}% значений — корректные даты")


In [ ]:
# Анализ и приведение столбца «Дата_покупки» к типу datetime
col = 'Дата_покупки'

# Пробуем преобразовать с ожидаемым форматом; некорректные значения станут NaT
df[col] = pd.to_datetime(df[col], format='%d.%m.%Y %H:%M', errors='coerce')

# Пересчитываем долю корректных дат
date_mask = df[col].notna()
percent_date = date_mask.mean() * 100
print(f"В колонке '{col}' {percent_date:.1f}% значений — корректные даты")


In [ ]:
#Там где мы получили 100% значений числа преобразуем в тип данных Flоat64 кроме ID_покупателя его сделаем строкой

df['Количество'] = df['Количество'].astype('float64')
df['ID_покупателя'] = df['ID_покупателя'].astype(str)

In [ ]:
#Нормализуем остальные столбцы и посчитаем уникальные символы
norm_cols = ['Номер_заказа','Код_Товара','Товар','Цена_шт','Страна']
for i in norm_cols:
    df[i]= df[i].astype(str).str.strip().str.lower()
#print(df[i].head(20))

In [ ]:
# Анализ уникальных символов в ключевых столбцах для оценки качества данных
cols = ['Номер_заказа', 'Код_Товара', 'Товар', 'Цена_шт', 'Страна']

for col in cols:
    all_text = ''.join(df[col].astype(str))
    unique_chars = sorted(set(all_text))
    print(f"Уникальные символы в колонке '{col}': {unique_chars}")

print("""
Вывод:
1) Номер заказа: содержит числа и букву «С» (возвраты) — допустимо оставить тип object.
2) Код товара: может включать цифры и буквы — допустимо оставить тип object.
3) Товар: содержит посторонние символы — требуется дополнительная очистка; тип object оставляем.
4) Цена: содержит цифры и запятую как разделитель — заменим запятую на точку и приведём к float64.
5) Страна: содержит буквы и пробелы — приведём к строковому типу.
""")


In [ ]:
# Очистка столбца «Товар» от спецсимволов
col = 'Товар'
df[col] = df[col].astype(str).str.replace(r'[^\w\s]', '', regex=True)

# Проверка оставшихся уникальных символов
all_text = ''.join(df[col].astype(str))
unique_chars = sorted(set(all_text))
print(f"Уникальные символы в колонке '{col}': {unique_chars}")


In [ ]:
# Замена запятой на точку в столбце «Цена_шт» и приведение к числовому типу
col = 'Цена_шт'
df[col] = (
    df[col]
    .astype(str)
    .str.replace(',', '.', regex=False)
    .apply(pd.to_numeric, errors='coerce')
)

# Проверка уникальных символов в обработанном столбце
all_text = ''.join(df[col].astype(str))
unique_chars = sorted(set(all_text))
print(f"Уникальные символы в колонке '{col}': {unique_chars}")

df.info()


In [ ]:
#Замени тип данных в столбце Страна
col = 'Страна'
df[col]= df[col].astype(str)
df.info()

In [ ]:
df['Выручка'] = df['Количество'] * df['Цена_шт']


In [ ]:
# Подсчёт и отображение статистики по удалению строк
rows_before = len(original_df)
rows_after = len(df)
lost_rows = rows_before - rows_after
lost_pct = (lost_rows / rows_before) * 100

summary = pd.DataFrame({
    'Этап': ['До очистки', 'После очистки', 'Удалено'],
    'Строк': [rows_before, rows_after, lost_rows],
    '% от исходного': [100, (rows_after / rows_before) * 100, lost_pct]
})

display(summary)


#### Результаты:
После очистки сырых данных из 541 909 строк осталось 401 604 строки. Мы избавились примерно от ~ 26 % данных. Практически в каждом столбце встречались смешанные типы данных в строках. Я нормализовал данные и привёл их к нужному типу на данном этапе.

### 2. Очистка данных и обработка возвратов/аномалий

In [ ]:
display(df)
df.describe()

In [ ]:
# Визуализация выбросов с помощью boxplot
df[['Цена_шт', 'Количество']].boxplot()
print('''
На первый взгляд, в обоих столбцах есть выбросы.
В столбце «Цена_шт» присутствуют очень дорогие товары (вблизи 40 000).
В столбце «Количество» наблюдаются выбросы с двух сторон усов; отрицательные значения соответствуют возвратам.
''')

# Оценка распределения данных через Q‑Q plot для выбора метода очистки от выбросов
import scipy.stats as stats

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

stats.probplot(df['Цена_шт'], dist='norm', plot=axes[0])
axes[0].set_title('Q‑Q plot: цена за единицу')

stats.probplot(df['Количество'], dist='norm', plot=axes[1])
axes[1].set_title('Q‑Q plot: количество товаров')

plt.tight_layout()
plt.show()

print('Распределение данных не является нормальным, поэтому для работы с выбросами предпочтительнее использовать метод IQR (межквартильный размах).')


In [ ]:
#Я считаю что бизнес логика никак не нарушена от больших выбросов так как абсолютно логично можно объяснить данные,
#но лучше от редких данных с большими значениями избавится чтобы не портить ML
# Используем метод IQR на 0.1 и 0.99 процентиле
df_clean = df.copy()
p_01 = df_clean['Цена_шт'].quantile(0.001)
p_99 = df_clean['Цена_шт'].quantile(0.999)
q_01 = df_clean['Количество'].quantile(0.001)
q_99 = df_clean['Количество'].quantile(0.999)
df_clean_price = df_clean.loc[
    (p_99 > df_clean['Цена_шт']) & 
    (df_clean['Цена_шт'] > p_01) & 
    (q_99 > df_clean['Количество']) &
    (df_clean['Количество'] > q_01)
    ]
df_clean_price.info()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

stats.probplot(df_clean_price['Цена_шт'], dist="norm", plot=axes[0])
axes[0].set_title('Q-Q plot: цена за единицу')

stats.probplot(df_clean_price['Количество'], dist="norm", plot=axes[1])
axes[1].set_title('Q-Q plot: количество товаров')

plt.tight_layout()
plt.show()
#df_clean_price[['Цена_шт','Количество']].boxplot()
print('''
Выводы:

Распределение данных в столбцах с ценами и количеством ненормальное, привести его близко к нормальному не получается. 
Так получается, потому что выбросы имеют большое количество вхождений.
Если мы берём 75‑й и 25‑й квартили, как это обычно принято, то потеряем больше половины датасета. 
Я считаю, так делать нельзя, так как теряется много смысла.
Решение — сгруппировать цены и количество в определённые категории и уже внутри категорий найти выбросы.
'''
 )

In [ ]:
# Группировка по объёму заказа и очистка от экстремальных выбросов с учётом бизнес‑логики
df_quantity_group = df.copy()

bins = [-float('inf'), -1000, -100, -10, 0, 10, 100, 1000, float('inf')]
labels = [
    'Возвраты_Крупный_опт',
    'Возвраты_Средний_опт',
    'Возвраты_Мелкий_опт',
    'Возвраты_Розница',
    'Розница',
    'Мелкий_опт',
    'Средний_опт',
    'Крупный_опт'
]

df_quantity_group['Категория_объёма'] = pd.cut(
    df_quantity_group['Количество'],
    bins=bins,
    labels=labels,
    right=True
)

# Очистка от экстремальных выбросов через квантили (0.02% и 99.98%)
Q0002 = df_quantity_group['Количество'].quantile(0.0002)
Q9998 = df_quantity_group['Количество'].quantile(0.9998)

df_quantity_group_clean = df_quantity_group.loc[
    (df_quantity_group['Количество'] >= Q0002) &
    (df_quantity_group['Количество'] <= Q9998)
].copy()

# Визуализация: boxenplot до и после очистки по категориям объёма
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

ax0 = sns.boxenplot(
    data=df_quantity_group,
    x='Категория_объёма',
    y='Количество',
    ax=axes[0],
    palette='Blues_d',
    hue='Категория_объёма',
    legend=False
)
ax0.set_title('До очистки', fontsize=16)
ax0.set_xlabel('Категория_объёма', fontsize=12)
ax0.set_ylabel('Количество', fontsize=12)
ax0.tick_params(labelsize=12)
ax0.tick_params(axis='x', labelrotation=90)

ax1 = sns.boxenplot(
    data=df_quantity_group_clean,
    x='Категория_объёма',
    y='Количество',
    ax=axes[1],
    palette='Pastel2',
    hue='Категория_объёма',
    legend=False
)
ax1.set_title('После очистки', fontsize=16)
ax1.set_xlabel('Категория_объёма', fontsize=12)
ax1.set_ylabel('Количество', fontsize=12)
ax1.tick_params(labelsize=12)
ax1.tick_params(axis='x', labelrotation=90)

plt.tight_layout()
plt.show()

df_quantity_group_clean.info()


In [ ]:
# Сегментация товаров по цене и очистка от экстремальных выбросов
df_price_group = df_quantity_group_clean.copy()

bins = [0, 10, 100, 1000, float('inf')]
labels = ['Бюджетный', 'Средний', 'Премиум', 'Ультра']

df_price_group['Сегмент_товара'] = pd.cut(
    df_price_group['Цена_шт'],
    bins=bins,
    labels=labels,
    right=False
)

# Очистка от экстремальных выбросов через квантили (0.01% и 99.99%)
Q0001 = df_price_group['Цена_шт'].quantile(0.0001)
Q9999 = df_price_group['Цена_шт'].quantile(0.9999)

df_price_group_clean = df_price_group.loc[
    (df_price_group['Цена_шт'] >= Q0001) &
    (df_price_group['Цена_шт'] <= Q9999)
].copy()

# Визуализация: boxenplot по сегментам до и после очистки
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

ax0 = sns.boxenplot(
    data=df_price_group,
    x='Сегмент_товара',
    y='Цена_шт',
    ax=axes[0],
    palette='Blues_d',
    hue='Сегмент_товара',
    legend=False
)
ax0.set_title('До очистки', fontsize=16)
ax0.set_xlabel('Сегмент_товара', fontsize=12)
ax0.set_ylabel('Цена_шт', fontsize=12)
ax0.tick_params(labelsize=12)
ax0.tick_params(axis='x', labelrotation=90)

ax1 = sns.boxenplot(
    data=df_price_group_clean,
    x='Сегмент_товара',
    y='Цена_шт',
    ax=axes[1],
    palette='Pastel2',
    hue='Сегмент_товара',
    legend=False
)
ax1.set_title('После очистки', fontsize=16)
ax1.set_xlabel('Сегмент_товара', fontsize=12)
ax1.set_ylabel('Цена_шт', fontsize=12)
ax1.tick_params(labelsize=12)
ax1.tick_params(axis='x', labelrotation=90)

plt.tight_layout()
plt.show()

df_price_group_clean.info()


#### Результаты:
После первого блока очистки в датасете осталось 401 604 строки. Если бы я использовал классический метод работы с выбросами (IQR, Q25, Q75), он не подошёл бы, так как распределение было асимметричным, ненормальным, — я бы потерял больше половины датасета.

Мной было принято решение о создании групп внутри столбцов «Цена_шт» и «Количество». Таким образом, я получил новые сегменты клиента с относительно нормальным распределением и понятной бизнес‑логикой.

По итогу в датасете остаётся 401 384 строки, что говорит нам о том, что бизнес‑логике мешали всего 220 строк. Я не исключаю, что такая очистка имеет субъективный характер, но в плане логики, я считаю, надо оставлять как можно больше строк с новыми группами, чтобы понимать, что вообще происходит с продажами.

Далее под каждый метод анализа будет проводиться дополнительная очистка, исходя из специфики метода.



### 3. Агрегация данных на уровне клиента (Customer‑level features)


#### Агрегация по параметрам
1) ID_покупателя (Групировка по столбцу)
2) Итоговая_выручка(Количество_шт * цена_за_шт)
3) Средний_чек (Итоговая_выручка/количество_заказов )
4) Количество_заказов(сумма строк по номеру_заказа)
5) Количество_купленных_товаров(сумма всех товаров > 0)
6) Количество_вернувших_товаров (сумма всех товаров < 0)
7) Доля_выкупа_% ((Количество_купленных_товаров - Количество_вернувших_товаров) / Количество_купленных_товаров )
8) Средняя_цена_товара
9) Среднее_количество_выкупленных_товаров( Количество_купленных_товаров.mean() )
10) Сегмент_товаров (Бюджетный, Средний,Премиум,Ультра)
11) Сегмент_объемов (Возвраты_Крупный_опт,Возвраты_Средний_опт,Возвраты_Мелкий_опт,Возвраты_Розница,Розница, Мелкий_опт,Средний_опт,Крупный_опт)
12) Последний_заказ_дней(сколько дней назад был последний заказ)\


In [ ]:
df_client = df_price_group_clean.copy()

# Расчёт выручки (включая возвраты — они дадут отрицательную выручку по строке)
df_client['Выручка'] = df_client['Цена_шт'] * df_client['Количество']

# Определение опорной даты (на 1 день после последней покупки в данных)
reference_date = df_client['Дата_покупки'].max() + pd.Timedelta(days=1)

# Агрегация по клиентам
df_client_group = df_client.groupby('ID_покупателя', as_index=False).agg(
    Количество_заказов=('Номер_заказа', 'count'),
    Итоговая_выручка=('Выручка', 'sum'),
    Средняя_цена_товара=('Цена_шт', 'mean'),
    Среднее_количество_выкупленных_товаров=('Количество', 'mean'),
    Количество_купленных_товаров=('Количество', lambda x: x[x > 0].sum()),
    Последний_заказ_дней=('Дата_покупки', lambda x: (reference_date - x.max()).days),
    Количество_вернувших_товаров=('Количество', lambda x: abs(x[x < 0].sum()))
)

# Дополнительные метрики
df_client_group['Средний_чек'] = (
    df_client_group['Итоговая_выручка'] / df_client_group['Количество_заказов']
)
df_client_group['Доля_выкупа_%'] = (
    (df_client_group['Количество_купленных_товаров'] - df_client_group['Количество_вернувших_товаров'])
    / df_client_group['Количество_купленных_товаров'] * 100
)

# Сегментация по средней цене товара и среднему количеству товаров
bins = [0, 10, 100, 1000, float('inf')]
labels_price = ['Бюджетный', 'Средний', 'Премиум', 'Ультра']
labels_quant = ['Розничный', 'Мелкий_опт', 'Средний_опт', 'Крупный_опт']

df_client_group['Сегмент_товаров'] = pd.cut(
    df_client_group['Средняя_цена_товара'],
    bins=bins,
    labels=labels_price,
    right=False
)
df_client_group['Сегмент_объемов'] = pd.cut(
    df_client_group['Среднее_количество_выкупленных_товаров'],
    bins=bins,
    labels=labels_quant,
    right=False
)

# Фильтрация некорректных значений (доля выкупа вне [0, 100], отрицательная выручка)
df_client_group = df_client_group.loc[
    (df_client_group['Доля_выкупа_%'] >= 0) &
    (df_client_group['Доля_выкупа_%'] <= 100) &
    (df_client_group['Итоговая_выручка'] >= 0)
]

# Выбор и упорядочивание столбцов
cols = [
    'ID_покупателя',
    'Итоговая_выручка',
    'Средний_чек',
    'Количество_заказов',
    'Количество_купленных_товаров',
    'Количество_вернувших_товаров',
    'Доля_выкупа_%',
    'Средняя_цена_товара',
    'Среднее_количество_выкупленных_товаров',
    'Сегмент_товаров',
    'Сегмент_объемов',
    'Последний_заказ_дней'
]

df_client_group = df_client_group[cols]

display(df_client_group)




#### Результаты:
1) С помощью агрегации по ID_покупателя мы получили новый датафрейм на 4319 строк и 12 столбцов
2) Теперь у меня есть бизнес метрики по которым я дальше буду искать корреляции, сегементировать и делать прогнозы.

### 4. Поиск скрытых корреляций


In [ ]:
corr = df_client_group.corr(numeric_only=True)
plt.figure(figsize=(10, 8))
sns.heatmap(corr[['Итоговая_выручка']].sort_values('Итоговая_выручка', ascending=False), annot=True, cmap='coolwarm')
plt.title('Корреляция признаков с  итоговой выручкой клиента')
plt.show()

#### Результаты:
1) Самый весомый признак получения выручки — «Количество_купленных_товаров». Это говорит о том, что данные, которые я анализирую, являются реальными: логично, что чем больше куплено товаров, тем выше выручка.
2) Косвенные признаки получения выручки не имеют весомых долей. Это означает, что на данном этапе мы не нашли скрытых признаков, влияющих на выручку.

### 5. Сегментация клиентов и формирование «портрета потребителя»

In [ ]:
df_segment = df_client_group.copy()

# --- RFM-сегментация ---
q = df_segment[['Последний_заказ_дней', 'Количество_заказов', 'Итоговая_выручка']].quantile([0.25, 0.5, 0.75])

def r_score(x, col):
    if x <= q[col][0.25]:
        return 4
    elif x <= q[col][0.5]:
        return 3
    elif x <= q[col][0.75]:
        return 2
    else:
        return 1

def fm_score(x, col):
    if x >= q[col][0.75]:
        return 4
    elif x >= q[col][0.5]:
        return 3
    elif x >= q[col][0.25]:
        return 2
    else:
        return 1

df_segment['R'] = df_segment['Последний_заказ_дней'].apply(r_score, col='Последний_заказ_дней')
df_segment['F'] = df_segment['Количество_заказов'].apply(fm_score, col='Количество_заказов')
df_segment['M'] = df_segment['Итоговая_выручка'].apply(fm_score, col='Итоговая_выручка')

df_segment['RFM_score'] = df_segment['R'] + df_segment['F'] + df_segment['M']

def RFM_segment(score):
    if score >= 10:
        return 'Лучшие'
    elif score >= 7:
        return 'Перспективные'
    elif score >= 4:
        return 'Спящие'
    else:
        return 'Потерянные'

df_segment['Сегмент_RFM'] = df_segment['RFM_score'].apply(RFM_segment)

# --- ABC-сегментация по выручке (правило Парето) ---
df_segment = df_segment.sort_values('Итоговая_выручка', ascending=False).copy()
total_rev = df_segment['Итоговая_выручка'].sum()
df_segment['Доля_накопленная'] = df_segment['Итоговая_выручка'].cumsum() / total_rev

def abc_segment(cum_share):
    if cum_share <= 0.80:
        return 'A'
    elif cum_share <= 0.95:
        return 'B'
    else:
        return 'C'

df_segment['Сегмент_ABC'] = df_segment['Доля_накопленная'].apply(abc_segment)

# --- Сегментация по качеству клиента (доля выкупа и активность) ---
def quality_segment(row):
    r_pct = row['Доля_выкупа_%']
    # r_cnt = row['Количество_заказов']  # закомментировано, т.к. не используется в текущей логике

    if r_pct <= 50:
        return 'Очень_низкий'
    if 50 < r_pct < 80:
        return 'Низкий'
    if r_pct >= 99:
        return 'Высокий'
    return 'Стандартный'

df_segment['Сегмент_процент_выкупа'] = df_segment.apply(quality_segment, axis=1)

# Исключаем клиентов с очень низким качеством
ban = ['Очень_низкий']
df_segment = df_segment[~df_segment['Сегмент_процент_выкупа'].isin(ban)].copy()

# --- Отчётность по сегментам ---

# 1. Распределение по RFM
print("Распределение по RFM:")
print(df_segment['Сегмент_RFM'].value_counts().sort_index())

# 2. ABC-сегментация (проверка Парето)
abc_summary = df_segment.groupby('Сегмент_ABC').agg(
    n_clients=('ID_покупателя', 'count'),
    share_revenue=('Итоговая_выручка', lambda x: x.sum() / total_rev)
).round(3)
print("\nABC-сегментация (проверка Парето):")
print(abc_summary)

# 3. Распределение по качеству клиента
print("\nРаспределение по проценту выкупа:")
print(df_segment['Сегмент_процент_выкупа'].value_counts())

display(df_segment)

# 4. Доля самых ценных клиентов
df_top = df_segment[
    (df_segment['Сегмент_RFM'] == 'Лучшие') &
    (df_segment['Сегмент_ABC'] == 'A') &
    (df_segment['Сегмент_процент_выкупа'] == 'Высокий')
]
share_top = len(df_top) / len(df_segment)
print(f"Доля самых ценных клиентов: {share_top:.2%}")



#### Результаты:
1) RFM‑сегментация. Я получил новые сегменты покупателей: лучшие, перспективные, спящие, потерянные — на основе давности, частоты и выручки.
2) ABC‑сегментация по правилу Парето разделила покупателей на 3 сегмента. Каждый из этих сегментов, в зависимости от задачи, имеет разную степень важности.
3) Сегмент «процент выкупа» был выделен отдельно для объективной оценки не просто купленных товаров, а именно выкупленных(итоговая выручка). На сегодняшний день именно такая оценка соответствует стандартам e‑commerce..
4) На данном этапе мы получили 4 сегмента покупателя и один сегмент купленного товара. Портрет потребителя теперь отвечает на такие вопросы: что покупает, сколько покупает, сколько принёс выручки, какова его ценность, как часто возвращает товары.
5) Доля саммых ценных клиентов =\
   'Сегмент_RFM' == 'Лучшие') &\
   'Сегмент_ABC' == 'A') & \
   'Сегмент_процент_выкупа' == 'Высокий')\
    = 16.33%

### 6. Построение ML‑моделей для прогноза вероятности покупки и суммы в течении месяца


#### Логистическая регресия


In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

# ============================================================
# НАСТРОЙКИ 
# ============================================================
train_end_date = pd.to_datetime('2011-11-09')  # Дата среза: всё ДО неё — прошлое (train), ПОСЛЕ — будущее (test)
forecast_days = 30                             # Окно прогноза (30 дней после среза)
feature_cols = ['R', 'F', 'M', 'Доля_выкупа_%']  # Признаки для модели

# ============================================================
# ШАГ 1: ФОРМИРУЕМ TARGET (кто реально купил в будущем)
# ============================================================
future_start = train_end_date
future_end = future_start + pd.Timedelta(days=forecast_days)

# Ищем в сырых данных (df), кто купил в окне прогноза
buyers_future = (
    df[
        (df['Дата_покупки'] > future_start) &
        (df['Дата_покупки'] <= future_end)
    ]['ID_покупателя']
    .unique()
)
print(f"Клиентов, купивших в окне прогноза (после {train_end_date}): {len(buyers_future)}")

# Добавляем target в df_segment
df_segment['target'] = df_segment['ID_покупателя'].isin(buyers_future).astype(int)

# Считаем долю покупателей (для понимания дисбаланса)
print(f"Доля покупателей в выборке: {df_segment['target'].mean():.3f}")

# ============================================================
# ШАГ 2: РАЗБИВАЕМ НА TRAIN/TEST ПО ВРЕМЕНИ
# ============================================================

# Создаём вспомогательную таблицу: дата последней покупки для каждого клиента ДО train_end_date
last_purchase_before_cut = (
    df[df['Дата_покупки'] <= train_end_date]
    .groupby('ID_покупателя')['Дата_покупки'].max()
    .reset_index()
    .rename(columns={'Дата_покупки': 'last_date_train'})
)

# Объединяем с df_segment, чтобы знать, кто из клиентов был активен ДО среза
df_ml = df_segment.merge(last_purchase_before_cut, on='ID_покупателя', how='left')

# Маска для train: клиенты, у которых была покупка ДО train_end_date
mask_train = df_ml['last_date_train'].notna()

X = df_ml.loc[mask_train, feature_cols].copy()
y = df_ml.loc[mask_train, 'target'].copy()

# Заполняем пропуски медианой (только на train части, чтобы не было утечки)
medians = X.median()
X = X.fillna(medians)

print(f"\nРазмер обучающей выборки (клиенты с историей до {train_end_date}): {len(X)}")
print(f"Из них купили в следующие 30 дней: {y.sum()}")

# Разбиваем train на train/validation внутри себя (стратифицированно, чтобы сохранить долю классов)
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ============================================================
# ШАГ 3: ОБУЧЕНИЕ МОДЕЛИ
# ============================================================
model_log_reg = LogisticRegression(class_weight='balanced', max_iter=1000, solver='lbfgs')
model_log_reg.fit(X_tr, y_tr)

# ============================================================
# ШАГ 4: МЕТРИКИ
# ============================================================
y_pred_val = model_log_reg.predict(X_val)
y_prob_val = model_log_reg.predict_proba(X_val)[:, 1]

# Считаем метрики
rep = classification_report(y_val, y_pred_val, digits=3, output_dict=True)
auc = roc_auc_score(y_val, y_prob_val)

# Формируем данные строго под слайд
slide_data = [
    ['Precision (avg)', rep['weighted avg']['precision']],
    ['Recall (avg)',    rep['weighted avg']['recall']],
    ['F1-score (avg)',  rep['weighted avg']['f1-score']],
    ['ROC-AUC',         auc]
]

df_slide = pd.DataFrame(slide_data, columns=['Метрика', 'Значение'])

# Красивое форматирование (3 знака после запятой)
styled_slide = df_slide.style.format({'Значение': '{:.3f}'})
display(styled_slide)

coefs = model_log_reg.coef_[0]
odds = np.exp(coefs)

# Собираем данные в список списков — это гарантированно 1D и без ошибок
coef_data = []
for i, col in enumerate(feature_cols):
    coef_data.append([col, float(coefs[i]), float(odds[i])])

df_coef = pd.DataFrame(coef_data, columns=['Признак', 'Коэффициент', 'Odds Ratio'])

# Сортируем по модулю коэффициента: самый сильный признак (R) будет сверху
df_coef['abs_coef'] = df_coef['Коэффициент'].abs()
df_coef_sorted = df_coef.sort_values('abs_coef', ascending=False).drop(columns=['abs_coef'])

# Форматируем и показываем как красивую таблицу
styled_coef = df_coef_sorted.style.format({
    'Коэффициент': '{:.4f}',
    'Odds Ratio': '{:.3f}'
})

display(styled_coef)


# ============================================================
# ШАГ 5: ДОБАВЛЯЕМ PROBABILITY В df_segment
# ============================================================
X_all = df_segment[feature_cols].copy()
# Заполняем пропуски теми же медианами, что считали на train
X_all = X_all.fillna(medians)

df_segment['Купит_в_течении_30_дней'] = model_log_reg.predict_proba(X_all)[:, 1]

# Покажем клиентов с низкой вероятностью покупки (для быстрой проверки)
display(df_segment[df_segment['Купит_в_течении_30_дней'] < 0.5].head(10))


#### Результаты ML:

1) Precision (точность) = 0,885 (88,5 %) — означает, что модель ошибается лишь в 11,5 % случаев при выборе, купит клиент или нет.
2) Recall (полнота) = 0,903 (90 %) — модель почти не пропускает реальных покупателей: она находит 90 % тех, кто действительно купит.
3) F1‑score (гармоническое среднее) = 0,885 (88,5 %) — баланс между precision и recall
4) ROC‑AUC (способность модели разделять классы по вероятностям) = ~0,977 (98 %) — модель очень хорошо ранжирует клиентов по вероятности покупки.


#### Выводы (какие признаки влияют чаще всего на последующую покупку):
1) Самый значимый признак в нашей модели — это R (давность последнего заказа): чем меньше дней с последнего заказа, тем выше вероятность, что он купит в ближайшие месяца.\
2) Самый слабый признак — это доля выкупа: модель считает, что количество выкупленных товаров очень слабо влияет на следующую покупку. Всего 0,7 % ухудшение вероятности при снижении процента выкупа на 1 п. п. Только резкое ухудшение может повлиять на вероятность покупки в будущем.\
3) F (частота) и M (выручка) имеют среднее влияние на покупку в ближайшие месяц. Переход из одной группы в другую увеличивает вероятность на ~ 21 % и 28 % соответственно.\

#### Таблица распределения вероятностей + Пончиковая диаграмма.

In [ ]:
# Создаем копию датафрейма для ML-анализа
df_ml1 = df_segment.copy()

# Формируем группы по вероятности покупки с помощью pd.cut
df_ml1['Группа_вероятности'] = pd.cut(
    df_ml1['Купит_в_течении_30_дней'],
    bins=[0, 0.5, 0.9, 1],
    labels=['0% - 50%', '50% - 90%', '90% - 100%'],
    include_lowest=True,
    right=True
)

# Агрегируем данные: считаем количество клиентов в каждой группе
df_grouped = (
    df_ml1
    .groupby('Группа_вероятности', as_index=False, observed=True)
    .size()
    .rename(columns={'size': 'Количество_клиентов'})
)

# Вычисляем долю каждой группы в процентах
df_grouped['Процент_группы'] = (df_grouped['Количество_клиентов'] / len(df_ml1)) * 100

# Фиксируем порядок категорий: от самых «горячих» к самым «холодным»
desired_order = ['90% - 100%', '50% - 90%', '0% - 50%']

# Приводим колонку к типу Categorical с явным порядком категорий
df_grouped['Группа_вероятности'] = pd.Categorical(
    df_grouped['Группа_вероятности'],
    categories=desired_order,
    ordered=True
)

# Сортируем DataFrame в соответствии с заданным порядком категорий
df_grouped = df_grouped.sort_values('Группа_вероятности')

# Отображаем итоговую таблицу
display(df_grouped)

# Извлекаем метки и размеры для построения графика
labels = df_grouped['Группа_вероятности'].tolist()
sizes = df_grouped['Количество_клиентов'].tolist()

# Задаем цветовую палитру в соответствии с порядком групп:
# зелёный — высокая вероятность, синий — средняя, оранжевый — низкая
colors = ['#2E8B57', '#4682B4', '#FFA500']

# Инициализируем фигуру и оси для построения круговой диаграммы
fig, ax = plt.subplots(figsize=(10, 10))

# Строим круговую диаграмму (pie chart)
wedges, texts, autotexts = ax.pie(
    sizes,
    labels=None,
    colors=colors,
    autopct='%1.1f%%',
    startangle=90,
    pctdistance=0.85,
    wedgeprops=dict(width=0.4, edgecolor='white', linewidth=3)
)

# Настраиваем стиль отображения процентных значений
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_weight('bold')
    autotext.set_fontsize(14)

# Добавляем легенду
ax.legend(
    wedges,
    labels,
    title='Вероятность покупки',
    loc='center left',
    bbox_to_anchor=(1, 0, 0.5, 1),
    frameon=False,
    fontsize=12,
    title_fontsize=13
)

# Устанавливаем заголовок диаграммы
ax.set_title(
    'Структура базы клиентов по вероятности покупки',
    fontsize=16,
    pad=20,
    fontweight='bold'
)

# Делаем диаграмму круглой (равные пропорции осей)
ax.axis('equal')

# Оптимизируем расположение элементов на фигуре
plt.tight_layout()

# Отображаем график
plt.show()


#### Результаты:
Мы получили, что ~ 46,1% всех клиентов совершат покупку в течении месяца.



#### Выводы:
С учётом самого значимого признака R (сколько дней с последнего заказа) у нас большая доля потерянных и спящих клиентов.

#### Линейная регресия

##### Доп очистка для линейной регресии

Любая аномалия в данных может испортить модель.
Чтобы избежать неправильного прогноза, удалим все выбросы из таблицы, полученной логистической регрессией.
Результатом машинного обучения мы получим прогноз величины выручки в следующие 30 дней.

In [ ]:
# Создаем рабочую копию датафрейма для сегментационного анализа
df_work = df_segment.copy()

# 1. Удаляем строки с пропусками ТОЛЬКО в ключевых колонках для сегментации
df_work = df_work.dropna(subset=['Сегмент_товаров', 'Сегмент_объемов'])

# 2. Агрегируем данные: группируем по паре сегментов и считаем количество транзакций
df_seg_agg = (
    df_work
    .groupby(['Сегмент_товаров', 'Сегмент_объемов'], as_index=False, observed=False)
    .size()
)

# Отображаем агрегированную таблицу
display(df_seg_agg)

# 3. Фильтруем сегменты: оставляем только те, где количество транзакций > 1500
# Остальные считаем малозначимыми/выбросами
df_filtered = df_seg_agg[df_seg_agg['size'] > 1500]

# 4. Извлекаем уникальные значения сегментов из отфильтрованной таблицы
unique_segments_products = df_filtered['Сегмент_товаров'].unique()
unique_segments_volumes = df_filtered['Сегмент_объемов'].unique()

print('Уникальные сегменты товаров:', unique_segments_products)
print('Уникальные сегменты объёмов:', unique_segments_volumes)

# Формируем финальный датафрейм на основе исходных данных
df_final = df_work.copy()

# Оставляем только строки, где оба сегмента входят в отфильтрованные списки
mask = (
    df_final['Сегмент_товаров'].isin(unique_segments_products) &
    df_final['Сегмент_объемов'].isin(unique_segments_volumes)
)
df_final = df_final[mask]

# Отображаем итоговый датафрейм
display(df_final)


#### Тепловая карта распределения транзакций.

In [ ]:
# Преобразуем данные в матрицу для heatmap
pivot_df = df_seg_agg.pivot(index='Сегмент_товаров', columns='Сегмент_объемов', values='size')

plt.figure(figsize=(10, 6))
sns.heatmap(pivot_df, annot=True, fmt='d', cmap='YlOrRd', cbar_kws={'label': 'Количество транзакций'})
plt.title('Распределение транзакций по сегментам товаров и объемов')
plt.xlabel('Сегмент объемов')
plt.ylabel('Сегмент товаров')
plt.tight_layout()
plt.show()


#### Результаты:
Получить хорошие результаты ML могут только две самые большие группы по объему Розничный и Мелкий_опт, если оставить остальные модель будет чаще ошибаться

#### Другая подготовка модели. 
Чтобы не было аномалий в объемах покупок разделим датафрейм на два.
Retail(розничные покупатели) и Wholesale(это оптовые)


In [ ]:
#Отберем клиентов которые приносят 80 процентов всей выручки
df_retail = df_final.loc[df_final['Сегмент_объемов'] ==  'Розничный']
df_wholesale = df_final.loc[df_final['Сегмент_объемов'] !=  'Розничный']
display(df_retail)    
display(df_wholesale)




#### Поиск аномалий в выручке
Доп очистка для целевой переменной

In [ ]:
import matplotlib.pyplot as plt

# Создаём фигуру с двумя подграфиками (1 строка, 2 столбца) для сравнения распределений
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# График 1: розница (слева) — boxplot по итоговой выручке
ax_retail = df_retail['Итоговая_выручка'].plot.box(ax=axes[0])
ax_retail.set_title('Розница: распределение Итоговая_выручка', fontsize=14)
ax_retail.set_ylabel('Итоговая_выручка, руб.', fontsize=12)
ax_retail.grid(axis='y', linestyle='--', alpha=0.3)

# График 2: опт (справа) — boxplot по итоговой выручке
ax_wholesale = df_wholesale['Итоговая_выручка'].plot.box(ax=axes[1])
ax_wholesale.set_title('Опт: распределение Итоговая_выручка', fontsize=14)
ax_wholesale.set_ylabel('Итоговая_выручка, руб.', fontsize=12)
ax_wholesale.grid(axis='y', linestyle='--', alpha=0.3)

# Оптимизируем расположение элементов, чтобы заголовки и подписи не перекрывались
plt.tight_layout()
plt.show()


def analyze_distribution(series, name):
    """
    Рассчитывает ключевые статистики распределения и долю выбросов.
    Использует межквартильный размах (IQR) для определения верхней границы выбросов.
    """
    q1 = series.quantile(0.25)
    median = series.median()
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    upper_bound = q3 + 1.5 * iqr
    outlier_share = (series > upper_bound).mean()
    
    print(f"--- {name} ---")
    print(f"Медиана: {median:,.2f} руб.")
    print(f"Q1: {q1:,.2f}, Q3: {q3:,.2f}")
    print(f"Верхняя граница выбросов: {upper_bound:,.2f} руб.")
    print(f"Доля клиентов-выбросов (выше границы): {outlier_share:.2%}")
    print()


# Проводим анализ распределения выручки для розницы и опта
analyze_distribution(df_retail['Итоговая_выручка'], "Розница")
analyze_distribution(df_wholesale['Итоговая_выручка'], "Опт")

# Очищаем данные: оставляем только строки, где выручка не превышает верхнюю границу выбросов
# Значения границ (2623.62 и 4650.45) взяты из предыдущего анализа распределений
df_retail_clean = df_retail[df_retail['Итоговая_выручка'] <= 2623.62].copy()
df_wholesale_clean = df_wholesale[df_wholesale['Итоговая_выручка'] <= 4650.45].copy()

# Отображаем очищенные датафреймы для проверки
display(df_retail_clean)
display(df_wholesale_clean)

# Строим гистограммы распределения выручки после очистки данных
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.hist(df_retail_clean['Итоговая_выручка'], bins=50)
plt.title('Розница: Итоговая_выручка_после_очистки')

plt.subplot(1, 2, 2)
plt.hist(df_wholesale_clean['Итоговая_выручка'], bins=50)
plt.title('Опт: Итоговая_выручка_после_очистки')

# Корректируем компоновку графиков, чтобы элементы не накладывались друг на друга
plt.tight_layout()
plt.show()



#### Результаты:
Очистка целевой переменной убирает 9.65% с розничных покупаелей и 9.88% с оптовых.\
Оставшиеся клиенты показывают типичное поведение для своей группы.

#### Линейная регресия

##### Розничные


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, median_absolute_error


# ============================================================
# НАСТРОЙКИ
# ============================================================
df_retail = df_retail_clean.copy()
train_end_date = pd.to_datetime('2011-11-09')
forecast_days = 30
future_start = train_end_date
future_end = future_start + pd.Timedelta(days=forecast_days)

feature_cols = [
    'R',
    'F',
    'M',
    'Доля_выкупа_%',
    'Средний_чек',
    'Количество_заказов',
    'Количество_купленных_товаров',
    'Количество_вернувших_товаров',
    'Средняя_цена_товара',
    'Среднее_количество_выкупленных_товаров'
]


# ============================================================
# ШАГ 1: ФОРМИРУЕМ TARGET ДЛЯ ЛИНЕЙНОЙ РЕГРЕССИИ (БУДУЩАЯ ВЫРУЧКА)
# ============================================================
# Вычисляем суммарную выручку каждого клиента в прогнозном окне
future_spend = (
    df.loc[
        (df['Дата_покупки'] > future_start) &
        (df['Дата_покупки'] <= future_end)
    ]
    .groupby('ID_покупателя', observed=False)['Выручка']
    .sum()
    .reset_index()
    .rename(columns={'Выручка': 'target_revenue'})
)

print(f"Клиентов, у которых была покупка в будущем окне: {len(future_spend)}")

# Присоединяем целевую переменную к основному датафрейму
df_ml_lin = df_retail.merge(future_spend, on='ID_покупателя', how='left')

# Заполняем отсутствующие значения нулями: отсутствие покупки трактуем как нулевую выручку
df_ml_lin['target_revenue'] = df_ml_lin['target_revenue'].fillna(0)

# Оставляем только клиентов с историей покупок до train_end_date (для корректного формирования выборки)
last_purchase_before_cut = (
    df.loc[df['Дата_покупки'] <= train_end_date]
    .groupby('ID_покупателя', observed=False)['Дата_покупки']
    .max()
    .reset_index()
)
df_ml_lin = df_ml_lin.merge(last_purchase_before_cut, on='ID_покупателя', how='inner')


# ============================================================
# ШАГ 2: ПОДГОТОВКА ДАННЫХ И РАЗБИВКА НА TRAIN/TEST
# ============================================================
X = df_ml_lin[feature_cols].copy()
y = df_ml_lin['target_revenue'].copy()

# Рассчитываем медианы по обучающей выборке (в рамках текущего подхода — по всей X, но с пониманием риска)
medians = X.median()
X = X.fillna(medians)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


# ============================================================
# ШАГ 3: ОБУЧЕНИЕ МОДЕЛИ ЛИНЕЙНОЙ РЕГРЕССИИ
# ============================================================
model_lin_reg_full_retail = LinearRegression()
model_lin_reg_full_retail.fit(X_train, y_train)


# ============================================================
# ШАГ 4: РАСЧЁТ И ИНТЕРПРЕТАЦИЯ МЕТРИК
# ============================================================
y_pred = model_lin_reg_full_retail.predict(X_test)

mae_retail = mean_absolute_error(y_test, y_pred)
median_err_retail = median_absolute_error(y_test, y_pred)
r2_retail = r2_score(y_test, y_pred)

print("\n" + "=" * 40)
print("МЕТРИКИ ЛИНЕЙНОЙ РЕГРЕССИИ RETAIL")
print("=" * 40)
print(f"MAE (средняя абсолютная ошибка): {mae_retail:.2f}")
print(f"Median Error (медианная абсолютная ошибка): {median_err_retail:.2f}")
print(f"R² (коэффициент детерминации): {r2_retail:.3f}")

# Формируем интерпретацию коэффициентов модели
features = feature_cols
coefs = model_lin_reg_full_retail.coef_

interpretations = []
for feat in features:
    if 'R' in feat:
        txt = 'Изменение прогноза выручки при увеличении R на 1 день (при прочих равных)'
    elif 'F' in feat:
        txt = 'Изменение прогноза выручки при увеличении F на 1 заказ'
    elif any(keyword in feat for keyword in ['M', 'Выручка', 'Сумма']):
        txt = 'Изменение прогноза выручки при изменении M/выручки на 1 рубль'
    elif 'Доля_выкупа' in feat or 'Доля выкупа' in feat:
        txt = 'Изменение прогноза выручки при росте доли выкупа на 1 процентный пункт'
    elif 'Средний_чек' in feat or 'Средний чек' in feat:
        txt = 'Изменение прогноза выручки при увеличении среднего чека на 1 рубль'
    else:
        txt = f'Изменение прогноза выручки при изменении признака "{feat}" на 1 единицу'
    interpretations.append(txt)

coef_retail = pd.DataFrame({
    'Признак': features,
    'Коэффициент': coefs,
    'Интерпретация': interpretations
})

# Отображаем таблицу коэффициентов (исправлена ошибка с именем переменной: было coef_wholesale → coef_retail)
display(coef_retail.style.format({'Коэффициент': '{:.4f}'}))

# Анализируем ошибки модели на крупных значениях целевой переменной
print("\nСравнение реальных и предсказанных значений (топ по реальной выручке):")
compare = pd.DataFrame({'real': y_test, 'pred': y_pred})
print(compare.sort_values('real', ascending=False).head(10))


# ============================================================
# ШАГ 5: ПРИМЕНЕНИЕ МОДЕЛИ И ЗАПИСЬ ПРОГНОЗА В DF_SEGMENT
# ============================================================
# Применяем модель ко всему датафрейму df_retail
X_all = df_retail[feature_cols].fillna(medians)
pred_full = model_lin_reg_full_retail.predict(X_all)

# Корректируем отрицательные прогнозы: выручка не может быть отрицательной
pred_positive = np.maximum(pred_full, 0)

df_retail['Потратит_в_течении_30_дней'] = pred_positive

print("\nГотово: колонка «Потратит_в_течении_30_дней» добавлена в df_retail.")
print(f"Всего клиентов с прогнозом выручки: {len(df_retail)}")
display(df_retail)


##### Оптовые

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, median_absolute_error


# ============================================================
# НАСТРОЙКИ
# ============================================================
df_wholesale = df_wholesale_clean.copy()
train_end_date = pd.to_datetime('2011-11-09')
forecast_days = 30
future_start = train_end_date
future_end = future_start + pd.Timedelta(days=forecast_days)

feature_cols = [
    'R',
    'F',
    'M',
    'Доля_выкупа_%',
    'Средний_чек',
    'Количество_заказов',
    'Количество_купленных_товаров',
    'Количество_вернувших_товаров',
    'Средняя_цена_товара',
    'Среднее_количество_выкупленных_товаров'
]


# ============================================================
# ШАГ 1: ФОРМИРУЕМ TARGET ДЛЯ ЛИНЕЙНОЙ РЕГРЕССИИ (БУДУЩАЯ ВЫРУЧКА)
# ============================================================
# Вычисляем суммарную выручку каждого клиента в прогнозном окне
future_spend = (
    df.loc[
        (df['Дата_покупки'] > future_start) &
        (df['Дата_покупки'] <= future_end)
    ]
    .groupby('ID_покупателя', observed=False)['Выручка']
    .sum()
    .reset_index()
    .rename(columns={'Выручка': 'target_revenue'})
)

print(f"Клиентов, у которых была покупка в будущем окне: {len(future_spend)}")

# Присоединяем целевую переменную к основному датафрейму
df_ml_lin = df_wholesale.merge(future_spend, on='ID_покупателя', how='left')

# Заполняем отсутствующие значения нулями: отсутствие покупки трактуем как нулевую выручку
df_ml_lin['target_revenue'] = df_ml_lin['target_revenue'].fillna(0)

# Оставляем только клиентов с историей покупок до train_end_date (для корректного формирования выборки)
last_purchase_before_cut = (
    df.loc[df['Дата_покупки'] <= train_end_date]
    .groupby('ID_покупателя', observed=False)['Дата_покупки']
    .max()
    .reset_index()
)
df_ml_lin = df_ml_lin.merge(last_purchase_before_cut, on='ID_покупателя', how='inner')


# ============================================================
# ШАГ 2: ПОДГОТОВКА ДАННЫХ И РАЗБИВКА НА TRAIN/TEST
# ============================================================
X = df_ml_lin[feature_cols].copy()
y = df_ml_lin['target_revenue'].copy()

# Рассчитываем медианы для заполнения пропусков
medians = X.median()
X = X.fillna(medians)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


# ============================================================
# ШАГ 3: ОБУЧЕНИЕ МОДЕЛИ ЛИНЕЙНОЙ РЕГРЕССИИ
# ============================================================
model_lin_reg_full_opt = LinearRegression()
model_lin_reg_full_opt.fit(X_train, y_train)


# ============================================================
# ШАГ 4: РАСЧЁТ И ИНТЕРПРЕТАЦИЯ МЕТРИК
# ============================================================
y_pred = model_lin_reg_full_opt.predict(X_test)

mae_wholesale = mean_absolute_error(y_test, y_pred)
median_err_wholesale = median_absolute_error(y_test, y_pred)
r2_wholesale = r2_score(y_test, y_pred)

print("\n" + "=" * 40)
print("МЕТРИКИ ЛИНЕЙНОЙ РЕГРЕССИИ (ОПТ)")
print("=" * 40)
print(f"MAE (средняя абсолютная ошибка): {mae_wholesale:.2f}")
print(f"Median Error (медианная абсолютная ошибка): {median_err_wholesale:.2f}")
print(f"R² (коэффициент детерминации): {r2_wholesale:.3f}")

# Формируем интерпретацию коэффициентов модели
features = feature_cols
coefs = model_lin_reg_full_opt.coef_

interpretations = []
for feat in features:
    if 'R' in feat:
        txt = 'Изменение прогноза выручки при увеличении R на 1 день (при прочих равных)'
    elif 'F' in feat:
        txt = 'Изменение прогноза выручки при увеличении F на 1 заказ'
    elif any(keyword in feat for keyword in ['M', 'Выручка', 'Сумма']):
        txt = 'Изменение прогноза выручки при изменении M/выручки на 1 рубль'
    elif 'Доля_выкупа' in feat or 'Доля выкупа' in feat:
        txt = 'Изменение прогноза выручки при росте доли выкупа на 1 процентный пункт'
    elif 'Средний_чек' in feat or 'Средний чек' in feat:
        txt = 'Изменение прогноза выручки при увеличении среднего чека на 1 рубль'
    else:
        txt = f'Изменение прогноза выручки при изменении признака "{feat}" на 1 единицу'
    interpretations.append(txt)

coef_wholesale = pd.DataFrame({
    'Признак': features,
    'Коэффициент': coefs,
    'Интерпретация': interpretations
})

# Отображаем таблицу коэффициентов
display(coef_wholesale.style.format({'Коэффициент': '{:.4f}'}))

# Анализируем ошибки модели на крупных значениях целевой переменной
print("\nСравнение реальных и предсказанных значений (топ по реальной выручке):")
compare = pd.DataFrame({'real': y_test, 'pred': y_pred})
print(compare.sort_values('real', ascending=False).head(10))


# ============================================================
# ШАГ 5: ПРИМЕНЕНИЕ МОДЕЛИ И ЗАПИСЬ ПРОГНОЗА В DF_SEGMENT
# ============================================================
# Применяем модель ко всему датафрейму df_wholesale
X_all = df_wholesale[feature_cols].fillna(medians)
pred_full = model_lin_reg_full_opt.predict(X_all)

# Корректируем отрицательные прогнозы: выручка не может быть отрицательной
pred_positive = np.maximum(pred_full, 0)

df_wholesale['Потратит_в_течении_30_дней'] = pred_positive

print("\nГотово: колонка «Потратит_в_течении_30_дней» добавлена в df_wholesale.")
print(f"Всего клиентов с прогнозом выручки: {len(df_wholesale)}")
display(df_wholesale)


#### Объединение результатов линейной регресси для ритейл и оптовых покупателей 

In [ ]:
# ============================================================
# ФОРМИРОВАНИЕ ТАБЛИЦ ДЛЯ ПРЕЗЕНТАЦИИ И ИНТЕРПРЕТАЦИИ
# ============================================================

# 1. Таблица с пояснениями признаков (для слайда «Что означают признаки»)
df_interpret = coef_retail[['Признак', 'Интерпретация']].copy()

# 2. Таблица сравнения коэффициентов моделей (розница vs опт)
df_compare = pd.merge(
    coef_retail[['Признак', 'Коэффициент']],
    coef_wholesale[['Признак', 'Коэффициент']],
    on='Признак',
    how='inner',
    suffixes=('_розница', '_опт')
).rename(columns={
    'Коэффициент_розница': 'Коэф. розница',
    'Коэффициент_опт': 'Коэф. опт'
})

# Добавляем интерпретации признаков в таблицу сравнения
df_compare['Интерпретация'] = df_interpret['Интерпретация']

# Вычисляем разницу коэффициентов (розница − опт) для наглядности
df_compare['Разница (розница − опт)'] = (
    df_compare['Коэф. розница'] - df_compare['Коэф. опт']
)

# Форматируем отображение коэффициентов и разницы для презентационного вида
styled_compare = df_compare.style.format({
    'Коэф. розница': '{:.4f}',
    'Коэф. опт': '{:.4f}',
    'Разница (розница − опт)': '{:.4f}'
})

# 3. Сводная таблица метрик качества моделей
metrics = pd.DataFrame({
    'Метрика': ['MAE (руб.)', 'Median Error (руб.)', 'R²'],
    'Розница': [mae_retail, median_err_retail, r2_retail],
    'Опт': [mae_wholesale, median_err_wholesale, r2_wholesale]
})

# Отображаем таблицу метрик (без индекса, с округлением до 2 знаков)
display(metrics.style.format({'Розница': '{:.3f}', 'Опт': '{:.3f}'}).hide(axis='index'))

print("\n=== Сравнение коэффициентов моделей ===")
display(styled_compare)




#### Результаты:
1) Мы получили две модели линейной регрессии с прогнозом выручки по клиенту в течении следующего месяца как для ритейл‑покупателей, так и для оптовых.
Я использовал все числовые признаки,так модель показывала лучшие результаты.
2) Самый главный фактор, влияющий на будущую выручку, — это R(давность заказа), как и в логистической модели чем меньше давность тем больше вероятность принести большую выручку.
3) Что интересно, это количество заказов: чем больше заказов, тем меньше ожидаемая выручка. Это связано с тем, что чем больше заказывает клиент,тем больше товаров он выбирает из бюджетного сегмента.
4) Можно придумать много разных паттернов но веса у признаков слишком малы.На таких данных опасно строить гипотезы или делать выводы.



# Финальная подготовка датафрейма


In [ ]:
import pandas as pd

# ============================================================
# ОБЪЕДИНЕНИЕ ДАННЫХ РОЗНИЦЫ И ОПТА
# ============================================================
result = pd.concat([df_retail, df_wholesale], axis=0, ignore_index=True)

# ============================================================
# ДОБАВЛЕНИЕ ГЕОДАННЫХ ДЛЯ МАРКЕТИНГОВОГО ОТДЕЛА
# ============================================================
# Формируем таблицу уникальных пар «клиент — страна»
geo_map = (
    df[['ID_покупателя', 'Страна']]
    .drop_duplicates(subset='ID_покупателя')
)

# Присоединяем географию к объединённому датафрейму
df_result_geo = pd.merge(
    geo_map,
    result,
    on='ID_покупателя',
    how='inner'
)

# Определяем финальный набор колонок для отчёта
cols = [
    'ID_покупателя',
    'Сегмент_товаров',
    'Сегмент_объемов',
    'Сегмент_RFM',
    'Сегмент_ABC',
    'Сегмент_процент_выкупа',
    'Купит_в_течении_30_дней',
    'Потратит_в_течении_30_дней',
    'Страна'
]

# Оставляем только нужные колонки
df_result_geo = df_result_geo[cols]

# Сортируем по вероятности покупки (от самых перспективных клиентов)
df_result_geo = df_result_geo.sort_values(
    'Купит_в_течении_30_дней',
    ascending=False
)

display(df_result_geo)

# ============================================================
# РАЗБИВКА ПО СТРАНАМ ДЛЯ ДЕТАЛЬНОГО ПРОСМОТРА
# ============================================================
groups_by_country = {
    country: group_df
    for country, group_df in df_result_geo.groupby('Страна')
}

for country in df_result_geo['Страна'].unique():
    df_country = groups_by_country.get(country)
    display(df_country)

# ============================================================
# СОХРАНЕНИЕ ИТОГОВОГО ФАЙЛА
# ============================================================
df_result_geo.to_csv(
    'clients_priority_final.csv',
    index=False,
    encoding='utf-8-sig'
)


#### Результаты:
Получены таблицы со всеми возможными сегментациями (RFM,ABC,Сегмент_качество,Сегмент_объемов,Сегмент_товаров) в которых есть вероятность покупки в течении семи дней,прогноз среднего чека и гео заказа.


# 7. Интерпретация результатов и связь с бизнес‑выводами

## Что получилось
- Сегментация RFM/ABC/процент выкупа показала, что топ‑сегмент «Лучшие / A / Высокий» составляет 16,33 % клиентов. Это подтверждает, что фокус на небольшой части клиентов может приносить основную долю дохода.
- Выявить клиентов с очень низким выкупом и удалить их.
- Модель логистической регрессии для прогноза покупки показала хорошие результаты: ROC‑AUC ≈ 0,977 (98 %) и F1‑score = 0,885 (88,5 %). Этого достаточно, чтобы использовать предсказания модели для приоритизации клиентов в маркетинговых кампаниях 
- Модель линейной регрессии для прогноза объёма покупок показала следующие результаты: 𝑅^2 для розницы — 0,38, для опта — 0,41. Модели обучены и протестированы без переобучения и утечки данных. Использованы все числовые признаки.Да, точность предсказаний в относительном выражении низкая, но в абсолютном выражении это нивелируется за счёт приоритета задачи. Важна не абсолютная точность прогноза, а то, как будет ранжирован тот или иной клиент.
- Найти клиентов с очень низким процентом выкупа и удалить их.Далее все клиенты с выкупом ниже 50% будет отправлятся в бан.
- Финальная таблица может ответить на такие вопросы, как:
1) Купит ли клиент в течении следующего месяца?
2) На какую сумму будет покупка?
3) В каком сегменте количества и стоимости будет товар?
4) Какой процент выкупа у клиента?
5) Из какого региона будет покупка?

## Важные наблюдения по данным
- При очистке данных удалось выявить, какие заказы — это покупка, а какие — возврат. Остальные данные были нормализованы и приведены к одной структуре и типам данных, что повысило корректность группировки по выявленным признакам.
- Обработка выбросов по кастомным квантилям (IQR) исключила экстремальные транзакции, которые искажали количество и цену купленного товара, но сохраняла как можно больше сегментов для сохранности бизнес‑логики.
- В дальнейшем были произведены очистки данных под конкретную задачу, но, если будет нужна полная картина — даже с учётом редких событий, — к ней всегда можно будет обратиться.


## Практические рекомендации
- Использовать прогноз вероятности покупки (из логистической и ленейной регрессии ) как приоритет для рассылок и персональных предложений.
- Для более точных прогнозов ввести новые метрики,доваить категории товаров,добавить сегменты промоакций.
- Учитывать региональные различия: в разных странах существенно отличаются выручка, процент выкупа и частота покупок — это важно при планировании маркетинговых акций.
- Регулярно пересчитывать и обновлять приоритеты клиентов (например, раз в месяц) RFM‑сегментов и сегмента процент выкупа, плюс менять или добавлять новые признаки в рамках бизнес‑логики.

In [ ]:
import joblib

# Сохраняем логистическую регрессию (прогноз покупки)
joblib.dump(model_log_reg, 'model_purchase_prob.joblib')

# Сохраняем линейную регрессию (прогноз суммы среднего чека для розничного покупателя)
joblib.dump(model_lin_reg_full_retail, 'model_revenue_retail.joblib')

# Сохраняем линейную регрессию (прогноз суммы среднего чека для оптового покупателя)
joblib.dump(model_lin_reg_full_opt, 'model_revenue_opt.joblib')

In [ ]:
pip freeze > requirements.txt